<a href="https://colab.research.google.com/github/IGLTrevo0/Flood-Segmentation/blob/main/Flood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

In [ ]:
torch.__version__


In [ ]:
import kagglehub
from google.colab import userdata
kagglehub.login()

In [ ]:


path = kagglehub.dataset_download("faizalkarim/flood-area-segmentation")
print(path)

In [ ]:
import os
for item in os.listdir(path):
  print(item)

In [ ]:
import pandas as pd
metadata_path = os.path.join(path, "metadata.csv")
df=pd.read_csv(metadata_path)
print(df.head())
print(df.tail())
print(df.shape)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image#PIL is for oppening and handelling images

row = df.iloc[0]
img_path = os.path.join(path, "Image", row["Image"])#this is bassically just path like how in computer its dekstop/folder etc its just like taht
mask_path = os.path.join(path, "Mask", row["Mask"])#as we are plotting this "row = df.iloc[0]"is taking first image and mask from folder

img = Image.open(img_path)
mask = Image.open(mask_path)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img)
ax[0].set_title("Image")
ax[1].imshow(mask, cmap="gray")
ax[1].set_title("Mask")
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T #converting images to tensors
import numpy as np

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)#when rows are radnomly choosen it gets messy .reset_index fixes it
val_df = val_df.reset_index(drop=True)
print(len(train_df), len(val_df))

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, dataframe, image_dir, mask_dir, transform=None, mask_transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["Image"])
        mask_path = os.path.join(self.mask_dir, row["Mask"])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        return image, mask

In [ ]:
IMG_SIZE = 128

image_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(), ])

mask_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

In [ ]:
image_dir = os.path.join(path, "Image")
mask_dir = os.path.join(path, "Mask")

train_dataset = FloodDataset(train_df, image_dir, mask_dir, image_transform, mask_transform)
val_dataset = FloodDataset(val_df, image_dir, mask_dir, image_transform, mask_transform)

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
imgs, masks = next(iter(train_loader))
print(imgs.shape, masks.shape)

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())
        self.pool2 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.Conv2d(128, 128, 3, padding=1), nn.ReLU())

        # Decoder
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())

        # Output
        self.out = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        u2 = self.up2(b)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return self.out(d1)

In [ ]:
model = SimpleUNet().to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def dice_coefficient(preds, targets, threshold=0.5, eps=1e-7):
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum(dim=(1,2,3))
    union = preds.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
    dice = (2 * intersection + eps) / (union + eps)
    return dice.mean().item()

In [ ]:
EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    val_dice = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            val_dice += dice_coefficient(outputs, masks)
    avg_val_loss = val_loss / len(val_loader)
    avg_val_dice = val_dice / len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Dice: {avg_val_dice:.4f}")

In [ ]:
model.eval()
imgs, masks = next(iter(val_loader))
imgs, masks = imgs.to(device), masks.to(device)

with torch.no_grad():
    preds = torch.sigmoid(model(imgs))
    preds = (preds > 0.5).float()

# move back to CPU + numpy for plotting
imgs_np = imgs.cpu().permute(0, 2, 3, 1).numpy()
masks_np = masks.cpu().squeeze(1).numpy()
preds_np = preds.cpu().squeeze(1).numpy()

fig, axes = plt.subplots(3, 4, figsize=(15, 10))
for i in range(4):
    axes[0, i].imshow(imgs_np[i]); axes[0, i].set_title("Image")
    axes[1, i].imshow(masks_np[i], cmap="gray"); axes[1, i].set_title("True Mask")
    axes[2, i].imshow(preds_np[i], cmap="gray"); axes[2, i].set_title("Predicted Mask")
    for ax in axes[:, i]: ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def evaluate_metrics(model, loader, device, threshold=0.5):
    model.eval()
    total_dice = 0
    total_correct = 0
    total_pixels = 0
    total_intersection = 0
    total_union_iou = 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            probs = torch.sigmoid(outputs)
            preds = (probs > threshold).float()

            total_dice += dice_coefficient(outputs, masks)


            correct = (preds == masks).float().sum()
            total_correct += correct.item()
            total_pixels += torch.numel(masks)

            intersection = (preds * masks).sum().item()
            union = ((preds + masks) >= 1).float().sum().item()
            total_intersection += intersection
            total_union_iou += union

    avg_dice = total_dice / len(loader)
    pixel_accuracy = total_correct / total_pixels
    iou = total_intersection / (total_union_iou + 1e-7)

    print(f"Final Pixel Accuracy: {pixel_accuracy*100:.2f}%")
    print(f"Final Dice Coefficient: {avg_dice*100:.2f}%")
    print(f"Final IoU Score: {iou*100:.2f}%")

evaluate_metrics(model, val_loader, device)